In [0]:
spark.table("workspace.logistics_project.fuel_purchases").printSchema()

root
 |-- fuel_purchase_id: string (nullable = true)
 |-- trip_id: string (nullable = true)
 |-- truck_id: string (nullable = true)
 |-- driver_id: string (nullable = true)
 |-- purchase_date: timestamp (nullable = true)
 |-- location_city: string (nullable = true)
 |-- location_state: string (nullable = true)
 |-- gallons: double (nullable = true)
 |-- price_per_gallon: double (nullable = true)
 |-- total_cost: double (nullable = true)
 |-- fuel_card_number: string (nullable = true)



In [0]:
#Read Tables
import pyspark.sql.functions as F

fuel = spark.table("workspace.logistics_project.fuel_purchases")
trips = spark.table("workspace.logistics_project.trips")
trucks = spark.table("workspace.logistics_project.trucks")

In [0]:
#Join Tables
fuel_analysis = (
    fuel
    .join(trips, ["trip_id","truck_id"], "inner")
    .join(trucks, "truck_id", "inner")
)

In [0]:
#Calculate Fuel KPIs
fuel_metrics = (
    fuel_analysis
    .groupBy(
        "truck_id",
        "unit_number",
        "make",
        "model_year",
        "fuel_type"
    )
    .agg(
        F.count("fuel_purchase_id").alias("fuel_purchases"),
        F.round(F.sum("gallons"),2).alias("total_gallons"),
        F.round(F.sum("total_cost"),2).alias("total_fuel_cost"),
        F.round(F.avg("price_per_gallon"),2).alias("avg_price_per_gallon"),
        F.round(F.avg("average_mpg"),2).alias("avg_mpg"),
        F.sum("actual_distance_miles").alias("total_miles")
    )
)

In [0]:
#Fuel Cost Per Mile
fuel_metrics = fuel_metrics.withColumn(
    "fuel_cost_per_mile",
    F.round(
        F.col("total_fuel_cost") /
        F.col("total_miles"),
        4
    )
)

In [0]:
#Miles Per Gallon Check
fuel_metrics = fuel_metrics.withColumn(
    "calculated_mpg",
    F.round(
        F.col("total_miles") /
        F.col("total_gallons"),
        2
    )
)

In [0]:
#Display Results
display(
    fuel_metrics.orderBy(
        F.col("fuel_cost_per_mile")
    )
)

truck_id,unit_number,make,model_year,fuel_type,fuel_purchases,total_gallons,total_fuel_cost,avg_price_per_gallon,avg_mpg,total_miles,fuel_cost_per_mile,calculated_mpg
TRK00106,5700,Volvo,2017,Diesel,2187,268563.2,1046859.73,3.89,6.51,3268617,0.3203,12.17
TRK00107,8740,Mack,2017,Diesel,2086,257768.4,1000715.53,3.89,6.5,3080191,0.3249,11.95
TRK00012,6381,Freightliner,2015,Diesel,2155,271078.5,1054377.25,3.89,6.53,3223597,0.3271,11.89
TRK00045,1715,Volvo,2015,Diesel,2013,249501.0,964414.79,3.87,6.5,2932899,0.3288,11.76
TRK00114,5193,Volvo,2016,Diesel,2112,258938.0,1008136.62,3.89,6.48,3059667,0.3295,11.82
TRK00054,4918,Freightliner,2016,Diesel,2045,254807.1,997212.37,3.91,6.48,3019839,0.3302,11.85
TRK00015,2531,International,2015,Diesel,2215,273774.3,1064716.48,3.89,6.51,3222096,0.3304,11.77
TRK00110,8349,Peterbilt,2016,Diesel,2100,260420.9,1010351.61,3.88,6.5,3058017,0.3304,11.74
TRK00041,3658,International,2015,Diesel,2202,270365.0,1059709.47,3.92,6.5,3200648,0.3311,11.84
TRK00057,4178,Mack,2015,Diesel,2202,272706.9,1069319.27,3.92,6.53,3226314,0.3314,11.83


Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
#Create Gold Table
spark.sql("""
SHOW TABLES IN workspace.logistics_gold
""").show(truncate=False)

+--------------+--------------------+-----------+
|database      |tableName           |isTemporary|
+--------------+--------------------+-----------+
|logistics_gold|driver_performance  |false      |
|logistics_gold|fleet_utilization   |false      |
|logistics_gold|maintenance_analysis|false      |
|logistics_gold|route_profitability |false      |
+--------------+--------------------+-----------+



In [0]:
fuel_metrics.write \
.format("delta") \
.saveAsTable(
    "workspace.logistics_gold.fuel_efficiency"
)

In [0]:
spark.sql("""
SHOW TABLES IN workspace.logistics_gold
""").show(truncate=False)

+--------------+--------------------+-----------+
|database      |tableName           |isTemporary|
+--------------+--------------------+-----------+
|logistics_gold|driver_performance  |false      |
|logistics_gold|fleet_utilization   |false      |
|logistics_gold|fuel_efficiency     |false      |
|logistics_gold|maintenance_analysis|false      |
|logistics_gold|route_profitability |false      |
+--------------+--------------------+-----------+



In [0]:
#MERGE
from delta.tables import DeltaTable

gold_table = DeltaTable.forName(
    spark,
    "workspace.logistics_gold.fuel_efficiency"
)

gold_table.alias("target").merge(
    fuel_metrics.alias("source"),
    "target.truck_id = source.truck_id"
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
#Verify
display(
    spark.table(
        "workspace.logistics_gold.fuel_efficiency"
    )
)

truck_id,unit_number,make,model_year,fuel_type,fuel_purchases,total_gallons,total_fuel_cost,avg_price_per_gallon,avg_mpg,total_miles,fuel_cost_per_mile,calculated_mpg
TRK00013,3093,Volvo,2018,Diesel,1957,240154.9,929763.05,3.87,6.5,2771115,0.3355,11.54
TRK00014,9624,International,2015,Diesel,2164,270239.7,1053249.21,3.9,6.53,3099507,0.3398,11.47
TRK00111,1086,Kenworth,2015,Diesel,2208,273080.7,1067902.85,3.91,6.47,3114489,0.3429,11.41
TRK00006,6082,Kenworth,2017,Diesel,2026,253486.9,987959.3,3.89,6.49,2955531,0.3343,11.66
TRK00083,8167,Kenworth,2015,Diesel,2015,248716.5,976805.64,3.92,6.5,2851879,0.3425,11.47
TRK00065,7859,Freightliner,2015,Diesel,2101,262187.3,1021752.12,3.9,6.48,2984322,0.3424,11.38
TRK00025,8967,Freightliner,2018,Diesel,2244,281631.0,1096847.71,3.89,6.49,3246146,0.3379,11.53
TRK00070,1854,Kenworth,2015,Diesel,2111,262426.4,1024084.35,3.9,6.52,2977457,0.3439,11.35
TRK00044,8170,Volvo,2015,Diesel,2283,287175.0,1117442.39,3.89,6.5,3332716,0.3353,11.61
TRK00063,9239,Peterbilt,2015,Diesel,2048,254904.2,987140.53,3.88,6.47,2905223,0.3398,11.4
